# Mage-Flow-Turbo on CPU
### Production-style Kaggle Demo with GGUF + stable-diffusion.cpp

> 🌐 Language / Ngôn ngữ: **English primary** · Vietnamese companion blocks use **Tiếng Việt** below.

Run **Mage-Flow-Turbo entirely on Kaggle CPU/RAM** with verified GGUF model inputs, a portable `stable-diffusion.cpp` runtime, a local REST API, and reproducible evidence.

**Default Run All:** one qualified 512×512 generation using the `demo` profile. Higher-resolution generation and the authenticated public endpoint are opt-in.

> **Tiếng Việt**
>
> Notebook này chạy **Mage-Flow-Turbo hoàn toàn bằng CPU/RAM trên Kaggle**, sử dụng các model GGUF đã xác minh, runtime `stable-diffusion.cpp` portable, REST API local và evidence có thể kiểm chứng lại.
>
> **Run All mặc định:** chỉ chạy một lần sinh ảnh 512×512 đã qualification với profile `demo`. Sinh ảnh độ phân giải cao hơn và public endpoint có xác thực đều là tùy chọn.

### 4 required Kaggle inputs

| Component | Purpose | Kaggle input / variation |
|---|---|---|
| Runtime | Verified prebuilt CPU `sd-cli` — recommended | `dangkhoa2016/stable-diffusion-cpp-6b3edaa-portable-cpu-runtime` |
| DiT | Mage-Flow-Turbo diffusion model | `dangkhoa2016/mage-flow-community-mage-flow-turbo` — **GGUF / q8-0** |
| VAE | Dedicated image decoder | `dangkhoa2016/mage-flow-community-mage-flow-turbo` — **PyTorch / vae-only** |
| Text encoder | Qwen3-VL 4B text encoder | `dangkhoa2016/qwen-qwen3-vl-4b-instruct-gguf` — **GGUF / q4-k-m** |

VAE SHA-256: `34e076dc1e8a15321e1e07be5111d59cf16dd10b804b7c7e20b4de29013427e0`

### What to expect

| Stage | Expected behavior |
|---|---|
| CPU runtime preparation | About **1.7 s** with the verified prebuilt runtime in the qualified release run |
| Source-build fallback | Historical preparation reference: **395.956 s (~396 s)** |
| Model verification | Full SHA-256 checks; may take noticeable time |
| 512×512 generation | **Several minutes** on shared Kaggle CPU |
| 640×640 | Optional and slower |
| 1024×1024 | Experimental / research; not part of the default qualified Run All |

This project demonstrates CPU portability and reproducibility; it is **not a GPU-class latency benchmark**. Vietnamese UTF-8 prompts are supported, but Vietnamese visual-quality parity is not claimed without a dedicated benchmark.

> **Tiếng Việt**
>
> Với runtime prebuilt đã attach, bước chuẩn bị runtime trong lần acceptance release mất khoảng **1,7 giây**; source-build lịch sử mất khoảng **395,956 giây (~396 giây)**. Sinh ảnh 512×512 trên CPU Kaggle vẫn mất **vài phút**. Mục tiêu của dự án là tính portable và khả năng tái lập, không phải tốc độ tương đương GPU.

## 0. Load a fresh copy of the repository

The Git checkout is disposable. Mutable runtime state stays outside the checkout under `/kaggle/working/mage-flow-turbo-runtime`.

`Restart Session → Run All` is the authoritative execution path. The notebook records the source repository used for the run.

> **Tiếng Việt**
>
> Git checkout chỉ là bản source tạm thời và có thể tạo lại. Runtime state được lưu riêng tại `/kaggle/working/mage-flow-turbo-runtime`.
>
> Quy trình chạy chính thức là `Restart Session → Run All`.

In [ ]:
from pathlib import Path
import os, shutil, subprocess

REPO_URL = "https://github.com/dangkhoa2016/Mage-Flow-Turbo-CPU.git"
if REPO_URL.startswith("__MAGE_"):
    raise RuntimeError("Repository origin has not been finalized. Run scripts/configure_repo_origin.py --apply in the real Git repository before authoritative Kaggle acceptance.")

repo_name = REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
REPO_DIR = Path('/kaggle/working') / repo_name
CLONE_TMP = REPO_DIR.with_name(REPO_DIR.name + '.clone-tmp')

git_env = os.environ.copy()
for key in ('GIT_ASKPASS', 'SSH_ASKPASS', 'GH_TOKEN', 'GITHUB_TOKEN'):
    git_env.pop(key, None)
for key in list(git_env):
    if key == 'GIT_CONFIG_COUNT' or key.startswith('GIT_CONFIG_KEY_') or key.startswith('GIT_CONFIG_VALUE_'):
        git_env.pop(key, None)
git_env['GIT_TERMINAL_PROMPT'] = '0'
git_env['GIT_CONFIG_GLOBAL'] = os.devnull
git_env['GIT_CONFIG_NOSYSTEM'] = '1'

# A stale temporary checkout is disposable. The final checkout is not touched
# until a replacement clone has succeeded and passed provenance checks.
if CLONE_TMP.exists():
    shutil.rmtree(CLONE_TMP)

clone_cmd = [
    'git',
    '-c', 'credential.helper=',
    '-c', 'http.extraHeader=',
    'clone', '--depth', '1', REPO_URL, str(CLONE_TMP),
]
subprocess.run(clone_cmd, check=True, env=git_env)

SOURCE_HEAD_CANDIDATE = subprocess.check_output(
    ['git', '-C', str(CLONE_TMP), 'rev-parse', 'HEAD'],
    text=True,
    env=git_env,
).strip()
SOURCE_STATUS_CANDIDATE = subprocess.check_output(
    ['git', '-C', str(CLONE_TMP), 'status', '--porcelain'],
    text=True,
    env=git_env,
)
if SOURCE_STATUS_CANDIDATE.strip():
    raise RuntimeError('Fresh source checkout is unexpectedly dirty; refusing to promote it.')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
CLONE_TMP.rename(REPO_DIR)

os.chdir(REPO_DIR)
SOURCE_HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if SOURCE_HEAD != SOURCE_HEAD_CANDIDATE:
    raise RuntimeError(
        f'Source HEAD changed during checkout promotion: candidate={SOURCE_HEAD_CANDIDATE} final={SOURCE_HEAD}'
    )
print('REPO_URL=', REPO_URL)
print('REPO_DIR=', REPO_DIR)
print('SOURCE_HEAD=', SOURCE_HEAD)
subprocess.run(['git', 'status', '--short'], check=True)


## 1. Configure the demo

The defaults are intentionally conservative:

- `RUN_LIVE_DEMO=True` — run exactly one canonical 512×512 generation.
- `ENABLE_PUBLIC_TUNNEL=False` — keep the service local by default.
- `RUN_OPTIONAL_USER_GENERATION=False` — custom generation is opt-in.
- `MAGE_PROFILE="demo"` — qualified default profile.
- `CUSTOM_PROFILE="balanced"` — suggested profile only when optional custom generation is enabled.

Local backend: `127.0.0.1:8090`  
Optional auth gateway: `127.0.0.1:8091`

> **Tiếng Việt**
>
> Cấu hình mặc định ưu tiên an toàn và khả năng tái lập: Run All chỉ sinh đúng một ảnh 512×512 chuẩn, public tunnel tắt, và phần sinh ảnh tùy chỉnh không chạy nếu người dùng chưa chủ động bật.

In [ ]:
RUN_LIVE_DEMO = True
ENABLE_PUBLIC_TUNNEL = False
RUN_OPTIONAL_USER_GENERATION = False
MAGE_PROFILE = "demo"
assert MAGE_PROFILE in {"demo", "balanced", "research"}

# Optional user-generation controls. These do not run unless RUN_OPTIONAL_USER_GENERATION=True.
CUSTOM_PROMPT = "A small red fox in a misty green forest, soft natural light, detailed photography."
CUSTOM_SEED = 43
CUSTOM_PROFILE = "balanced"
assert CUSTOM_PROFILE in {"demo", "balanced", "research"}
os.environ['MAGE_RUNTIME_ROOT'] = '/kaggle/working/mage-flow-turbo-runtime'
runtime_root = Path(os.environ['MAGE_RUNTIME_ROOT'])

# A Run All qualification must begin with fresh authoritative runtime state.
# Stop any stale owned processes first; the existing stop helpers fail closed
# if a generation is busy or an orphan sd-cli is detected.
subprocess.run(['bash','scripts/kaggle/stop-authenticated-public-demo.sh'],check=True)
subprocess.run(['bash','scripts/kaggle/stop-cpu-demo.sh'],check=True)
if runtime_root.exists():
    shutil.rmtree(runtime_root)
print('RELEASE_RUNTIME_STATE_RESET=PASS')
os.environ['MAGE_DEFAULT_PROFILE'] = 'demo'
os.environ['MAGE_BACKEND'] = 'cpu'
os.environ['ENABLE_PUBLIC_TUNNEL'] = 'True' if ENABLE_PUBLIC_TUNNEL else 'False'
print({
    "RUN_LIVE_DEMO": RUN_LIVE_DEMO,
    "ENABLE_PUBLIC_TUNNEL": ENABLE_PUBLIC_TUNNEL,
    "RUN_OPTIONAL_USER_GENERATION": RUN_OPTIONAL_USER_GENERATION,
    "MAGE_PROFILE": MAGE_PROFILE,
    "CUSTOM_PROFILE": CUSTOM_PROFILE,
    "CUSTOM_SEED": CUSTOM_SEED,
})

# PUBLIC PREBUILT RUNTIME DELIVERY
os.environ["MAGE_RUNTIME_MODE"] = "auto"
PUBLIC_RUNTIME_DATASET_SLUG = "dangkhoa2016/stable-diffusion-cpp-6b3edaa-portable-cpu-runtime"
PUBLIC_RUNTIME_DATASET_ROOT = Path("/kaggle/input/datasets/dangkhoa2016/stable-diffusion-cpp-6b3edaa-portable-cpu-runtime")
PUBLIC_RUNTIME_SD_CLI = Path("/kaggle/input/datasets/dangkhoa2016/stable-diffusion-cpp-6b3edaa-portable-cpu-runtime/stable-diffusion-cpp-6b3edaa-portable-cpu-runtime/sd-cli")
PUBLIC_RUNTIME_SD_CLI_SHA256 = "7539d90b99eaf2b6279eec4f9006a68ae53e87bfe0c9c325ff3f329220468a5c"

if PUBLIC_RUNTIME_SD_CLI.is_file():
    os.environ["MAGE_PREBUILT_SD_CLI"] = str(PUBLIC_RUNTIME_SD_CLI)
    os.environ["MAGE_PREBUILT_SD_CLI_SHA256"] = PUBLIC_RUNTIME_SD_CLI_SHA256
    print("PUBLIC_RUNTIME_PREBUILT_HINT=PASS")
else:
    os.environ.pop("MAGE_PREBUILT_SD_CLI", None)
    os.environ.pop("MAGE_PREBUILT_SD_CLI_SHA256", None)
    print("PUBLIC_RUNTIME_PREBUILT_HINT=NOT_ATTACHED_SOURCE_FALLBACK_AVAILABLE")
# END PUBLIC PREBUILT RUNTIME DELIVERY

## 2. Run static checks

Run the repository's unit, negative-contract, notebook, runtime, and publication checks **before** any real model inference.

These checks are inexpensive compared with CPU image generation and fail early if the source contract is inconsistent.

> **Tiếng Việt**
>
> Chạy toàn bộ static/fake/contract test trước khi inference thật. Nếu source hoặc contract có vấn đề, notebook sẽ dừng sớm thay vì tiêu tốn thời gian CPU để sinh ảnh.

In [ ]:
subprocess.run(['python3','-m','unittest','discover','-s','tests','-v'],check=True,env={**os.environ,'MAGE_ALLOW_REPO_PLACEHOLDER':'0'})


## 3. Prepare the CPU runtime

Pinned `stable-diffusion.cpp` commit:

`6b3edaaf32cc19e5bb2d819c788bd557eddc8eba`

Runtime policy is `auto`:

1. Prefer the verified prebuilt CPU `sd-cli` when the public runtime dataset is attached.
2. If the prebuilt runtime is unavailable, retain the pinned source-build fallback.
3. Verify the selected binary before use.

> **Tiếng Việt**
>
> Notebook ưu tiên `sd-cli` CPU prebuilt đã được xác minh. Nếu runtime dataset không được attach, chế độ `auto` vẫn giữ source-build fallback đã pin. Vì source build có thể mất nhiều phút, nên attach runtime dataset là lựa chọn khuyến nghị.

In [ ]:
subprocess.run(['bash','scripts/kaggle/capture-environment.sh'],check=True)
subprocess.run(['bash','scripts/kaggle/bootstrap-cpu-demo.sh'],check=True)


## 4. Verify the model files

Before starting the API, the preflight performs full SHA-256 verification for the canonical inputs, including the dedicated `vae-only` variation.

The resolver is fail-closed: ambiguous or incorrect model files stop the run.

> **Tiếng Việt**
>
> Trước khi khởi động API, preflight hash đầy đủ các model canonical, bao gồm variation `vae-only`. Nếu file sai hoặc resolver gặp nhiều candidate không rõ ràng, notebook sẽ dừng ngay.

In [ ]:
subprocess.run(['python3','scripts/kaggle/preflight.py'],check=True)
os.environ['MAGE_REUSE_PREFLIGHT'] = '1'  # Avoid hashing immutable attached inputs twice in the same Run All.


## 5. Start the local API

Start the CPU-only REST service and verify real readiness before sending any generation request.

The backend remains bound to `127.0.0.1:8090`.

> **Tiếng Việt**
>
> Khởi động REST API chạy CPU-only và kiểm tra readiness thật trước khi gửi request sinh ảnh. Backend mặc định chỉ bind local tại `127.0.0.1:8090`.

In [ ]:
subprocess.run(['bash','scripts/kaggle/start-cpu-demo.sh'],check=True)
subprocess.run(['bash','scripts/kaggle/status-cpu-demo.sh'],check=True)


## 6. Generate the first image

This is the **only automatic real generation** in the default Run All:

- Resolution: **512×512 — Recommended / qualified**
- Profile: `demo`
- Steps: `4`
- CFG: `1.0`
- Threads: `4`
- Seed: `42`

There is **no automatic retry after `sd-cli` begins**.

> **Tiếng Việt**
>
> Đây là inference thật duy nhất chạy tự động trong Run All mặc định: 512×512, profile `demo`, seed 42. Sau khi `sd-cli` bắt đầu chạy, notebook không tự động retry.

In [ ]:
if RUN_LIVE_DEMO:
    subprocess.run(['python3','scripts/kaggle/local_acceptance.py'],check=True)
else:
    print('CORE_LOCAL_DEMO=NOT_RUN')


## 7. View the result

Display the canonical 512×512 PNG produced by the local acceptance request.

The image is also retained in the evidence directory for hash and dimension verification.

> **Tiếng Việt**
>
> Hiển thị ảnh PNG 512×512 vừa sinh. Ảnh đồng thời được lưu trong evidence để kiểm tra SHA-256, kích thước và tính nhất quán với REST response.

In [ ]:
if RUN_LIVE_DEMO:
    from IPython.display import display
    from PIL import Image
    p=Path(os.environ['MAGE_RUNTIME_ROOT'])/'evidence/release-acceptance-512.png'
    display(Image.open(p))


## 8. Try a custom prompt or higher resolution

This cell is **inert during default Run All**. To use it, set `RUN_OPTIONAL_USER_GENERATION=True` in the configuration cell and edit:

- `CUSTOM_PROMPT`
- `CUSTOM_SEED`
- `CUSTOM_PROFILE`

| Profile | Resolution | Publication status |
|---|---:|---|
| `demo` | 512×512 | **Recommended / qualified** |
| `balanced` | 640×640 | **Optional** |
| `research` | 1024×1024 | **Experimental / research** |

The `research` profile also requires the stricter 20 GiB available-RAM / 3 GiB free-disk admission gate.

> **Tiếng Việt**
>
> Cell này không chạy trong Run All mặc định. Muốn sinh ảnh tùy chỉnh, bật `RUN_OPTIONAL_USER_GENERATION=True`, sau đó chỉnh prompt, seed và profile. 512×512 là cấu hình đã qualification; 640×640 là tùy chọn; 1024×1024 hiện được ghi rõ là experimental/research.

In [ ]:
if RUN_OPTIONAL_USER_GENERATION:
    import json
    from urllib.request import Request, urlopen
    payload={'prompt':CUSTOM_PROMPT,'seed':CUSTOM_SEED,'profile':CUSTOM_PROFILE}
    req=Request(
        'http://127.0.0.1:8090/v1/images/generate',
        data=json.dumps(payload,ensure_ascii=False).encode('utf-8'),
        headers={'Content-Type':'application/json'},
        method='POST',
    )
    http_timeout={'demo':1000,'balanced':1300,'research':2850}[CUSTOM_PROFILE]
    print(urlopen(req,timeout=http_timeout).read().decode('utf-8'))
else:
    print('OPTIONAL_USER_GENERATION=NOT_RUN')

## 9. Optional authenticated public endpoint

Public access is disabled by default.

When explicitly enabled, the tunnel points to the authenticated gateway on `127.0.0.1:8091`, **never directly to the backend on 8090**. Public acceptance reuses the existing canonical artifact and does not launch another `sd-cli` generation.

> **Tiếng Việt**
>
> Public endpoint mặc định tắt. Khi bật, tunnel chỉ trỏ tới auth gateway cổng 8091, không expose trực tiếp backend 8090. Public acceptance dùng lại artifact canonical đã có và không chạy thêm một inference mới.

In [ ]:
if ENABLE_PUBLIC_TUNNEL:
    subprocess.run(['bash','scripts/kaggle/start-authenticated-public-demo.sh'],check=True)
    subprocess.run(['python3','scripts/kaggle/public_acceptance.py'],check=True)
else:
    print('AUTHENTICATED_PUBLIC_DEMO=NOT_RUN')


## 10. Reproducibility report

Collect sanitized evidence for the completed run: environment, model/runtime identities, request/result metadata, telemetry, output image information, and clean-stop state.

Raw sensitive execution details remain excluded from the publication evidence contract.

> **Tiếng Việt**
>
> Thu thập evidence đã sanitize để có thể kiểm tra lại môi trường, danh tính model/runtime, request/result, telemetry, output image và trạng thái dừng service. Các chi tiết nhạy cảm không được đưa vào publication evidence.

In [ ]:
subprocess.run(['bash','scripts/kaggle/collect-production-demo-evidence.sh'],check=True)
print('EVIDENCE_COLLECTION=PASS')


## 11. Clean shutdown

Stop the optional authenticated gateway (if enabled) and stop the local CPU service. The stop helpers are idempotent, so cleanup remains safe even if evidence collection has already stopped the service.

> **Tiếng Việt**
>
> Dừng auth gateway nếu đã bật và dừng REST service local. Các script stop là idempotent nên có thể chạy an toàn ngay cả khi evidence collector đã dừng service trước đó.

In [ ]:
if ENABLE_PUBLIC_TUNNEL:
    subprocess.run(['bash','scripts/kaggle/stop-authenticated-public-demo.sh'],check=True)
subprocess.run(['bash','scripts/kaggle/stop-cpu-demo.sh'],check=True)


## 12. Final summary

A successful default Run All requires:

- canonical local generation: PASS
- sanitized evidence collection: PASS
- clean service lifecycle
- public tunnel: may remain `NOT_RUN`

The cell below first shows a human-readable summary, then preserves the machine-readable verification markers used by the forensic contracts.

> **Tiếng Việt**
>
> Run All mặc định được xem là thành công khi local generation và evidence collection đều PASS, service lifecycle sạch. Public tunnel có thể giữ trạng thái `NOT_RUN`. Cell cuối hiển thị summary dễ đọc trước, sau đó vẫn giữ nguyên các marker dành cho máy kiểm tra.

In [ ]:
import json
from IPython.display import Markdown, display

state_root=Path(os.environ['MAGE_RUNTIME_ROOT'])/'state'
local_path=state_root/'local-acceptance.json'
local_ok=local_path.exists() if RUN_LIVE_DEMO else False
evidence_ok=(state_root/'evidence-collection.json').exists()
public_state='PASS' if ENABLE_PUBLIC_TUNNEL and (state_root/'public-acceptance.json').exists() else 'NOT_RUN'

if local_ok:
    local_report=json.loads(local_path.read_text(encoding='utf-8'))
    artifact=local_report.get('artifact',{})
    elapsed_s=local_report.get('http_elapsed_ms',0)/1000.0
    friendly = f"""### ✅ Demo completed successfully

| Item | Result |
|---|---|
| Runtime policy | `auto` — verified prebuilt preferred |
| Profile | `{local_report.get('profile','demo')}` |
| Resolution | {artifact.get('width','?')}×{artifact.get('height','?')} |
| Seed | {local_report.get('seed','?')} |
| HTTP generation time | {elapsed_s:.1f} s |
| Image SHA-256 | `{artifact.get('sha256','?')}` |
| Evidence | {'Collected' if evidence_ok else 'Missing'} |
| Public endpoint | {public_state} |
"""
    display(Markdown(friendly))

print('--- machine-readable verification ---')
print('CORE_LOCAL_DEMO=' + ('PASS' if local_ok else 'FAIL'))
print('EVIDENCE_COLLECTION=' + ('PASS' if evidence_ok else 'FAIL'))
print('AUTHENTICATED_PUBLIC_DEMO=' + public_state)
final_ok=local_ok and evidence_ok
print('PRODUCTION_ORIENTED_DEMO_NOTEBOOK=' + ('PASS' if final_ok else 'INCOMPLETE'))
if not final_ok:
    raise RuntimeError('Production notebook is incomplete; inspect prior gates/evidence.')